In [51]:
import re
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, precision_recall_curve
import matplotlib.pyplot as plt
from matplotlib import lines

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [52]:
from pathlib import Path
ROOT = Path('..')
path = ROOT / 'data' / 'processed' / 'draft_enriched_with_contracts.csv'
raw = pd.read_csv(path, low_memory=False)
df = raw.query('2014 <= year <= 2025').copy()
text_cols = ['overview', 'strengths', 'weaknesses']
df[text_cols] = df[text_cols].fillna('')
df['scouting_text'] = (
    df[text_cols].agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
numeric_cols = ['grade', 'total_score', 'production_score', 'athleticism_score']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
df['Pos_Group'] = df['Pos_Group'].fillna('UNKNOWN')
df['made_it_contract'] = df['made_it_contract'].where(df.year <= 2021)
df = df.loc[(df['grade'] > 0) & (df['scouting_text'].str.strip() != '')].copy()

In [53]:
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide', 'pass',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}
CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'work', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}
PHRASE_BLOCKLIST = [
    'undrafted free agent', 'practice squad', 'free agent', 'early starter',
    'pro bowl', 'late round', 'undrafted free', 'make roster', 'rostered',
]
_base_stops = set(stopwords.words('english'))
NFL_STOPWORDS = (_base_stops - KEEP_WORDS) | CUSTOM_STOPS
_lemmatizer = WordNetLemmatizer()

def nfl_preprocess_no_stitch(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    for phrase in sorted(PHRASE_BLOCKLIST, key=len, reverse=True):
        text = text.replace(phrase, ' ')
    text = re.sub(r'[-–—]', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in NFL_STOPWORDS and len(t) > 1]
    tokens = [_lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

_tag_prefix = {'overview': 'ovr', 'strengths': 'str', 'weaknesses': 'wkn'}
for col, prefix in _tag_prefix.items():
    df[f'{col}_clean'] = df[col].apply(nfl_preprocess_no_stitch)

In [54]:
# ── Section-tagged token representation ──────────────────────────────────────
# Instead of "OVR: token1 token2 STR: token3 token4" (cross-boundary bigrams),
# prefix every token with its section: "ovr_token1 ovr_token2 str_token3 str_token4"
#
# Benefits:
#   - No cross-section boundary bigrams by construction
#   - Bigrams stay within sections: "str_quick str_off_snap"
#   - Every feature is directly interpretable: ovr_* vs str_* vs wkn_*
#   - Can analyse which section drives predictions for each player/position

def build_section_tagged(row):
    parts = []
    for col, prefix in _tag_prefix.items():
        clean = str(row[f'{col}_clean']).strip()
        if clean:
            tagged = [f"{prefix}_{tok}" for tok in clean.split()]
            parts.extend(tagged)
    return ' '.join(parts)

df['scouting_text_tagged'] = df.apply(build_section_tagged, axis=1)
df = df.loc[df['scouting_text_tagged'].str.strip() != ''].copy()

# Raw section token % — added as optional numeric features (no model dependency)
for col in ['overview', 'strengths', 'weaknesses']:
    df[f'{col}_ntok'] = df[f'{col}_clean'].str.split().str.len().fillna(0)
df['_total_ntok'] = (df['overview_ntok'] + df['strengths_ntok'] + df['weaknesses_ntok']).replace(0, 1)
df['str_tok_pct'] = df['strengths_ntok'] / df['_total_ntok']
df['wkn_tok_pct'] = df['weaknesses_ntok'] / df['_total_ntok']
df['ovr_tok_pct'] = df['overview_ntok']  / df['_total_ntok']

# Sanity check
token_counts = df['scouting_text_tagged'].str.split().str.len()
print(f'Players: {len(df)}, tokens/player median: {int(token_counts.median())}')
print(f'Mean str_tok_pct={df.str_tok_pct.mean():.2f}  wkn={df.wkn_tok_pct.mean():.2f}  ovr={df.ovr_tok_pct.mean():.2f}')
print()
sample = df['scouting_text_tagged'].iloc[0]
print('Sample (first 300 chars):')
print(sample[:300])

Players: 5321, tokens/player median: 130
Mean str_tok_pct=0.36  wkn=0.30  ovr=0.34

Sample (first 300 chars):
ovr_physical ovr_specimen ovr_rare ovr_size ovr_speed ovr_combination ovr_clowney ovr_impactful ovr_junior ovr_playing ovr_through ovr_injury ovr_forced ovr_deal ovr_opposing ovr_offense ovr_fully ovr_accounted ovr_extra ovr_chip ovr_protection ovr_old ovr_junior ovr_affected ovr_turnover ovr_defens


In [ ]:
# Four-way position split — same as two_group_tf_idf
BLOCK    = {'OL', 'DT'}
COVERAGE = {'EDGE', 'DB', 'LB'}
SKILL    = {'WR', 'RB', 'TE', 'UNKNOWN'}
QB_GRP   = {'QB'}

# max_features=1500: vocab is ~3x larger since each token has ovr/str/wkn prefix
# No stop_words needed — cross-boundary artifacts are impossible by design
vectorizer_params = dict(max_features=1500, ngram_range=(2, 3), min_df=3, sublinear_tf=True)
numeric_features  = ['grade', 'total_score', 'production_score', 'athleticism_score']
cat_features      = ['Pos_Group']

class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.columns]

def make_text_pipe():
    return Pipeline([
        ('selector', ColumnSelector(['scouting_text_tagged'])),
        ('squeeze',  FunctionTransformer(lambda x: x.squeeze())),
        ('tfidf',    TfidfVectorizer(**vectorizer_params)),
    ])

def make_meta_transformer():
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(),                       numeric_features),
            ('pos', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ],
        remainder='drop',
    )

def make_full_pipe():
    return Pipeline([
        ('features', FeatureUnion([
            ('text', make_text_pipe()),
            ('meta', make_meta_transformer()),
        ])),
        ('rf', RandomForestClassifier(n_estimators=400, class_weight='balanced',
                                      random_state=42, n_jobs=-1)),
    ])

def make_baseline_pipe():
    return Pipeline([
        ('preproc', ColumnTransformer(
            transformers=[
                ('grade', StandardScaler(),                       ['grade']),
                ('pos',   OneHotEncoder(handle_unknown='ignore'), ['Pos_Group']),
            ],
            remainder='drop',
        )),
        ('lr', LogisticRegression(max_iter=2000, solver='liblinear')),
    ])

def fit_baseline(pipe, X, y):
    pos_rate = y.mean()
    w = y*(0.3/pos_rate) + (1-y)*(0.7/(1-pos_rate))
    pipe.fit(X, y, lr__sample_weight=w)

def cv_baseline(pipe, X, y, cv):
    pos_rate = y.mean()
    pw, nw = 0.3/pos_rate, 0.7/(1-pos_rate)
    weights = y*pw + (1-y)*nw
    scores = []
    for tr, te in cv.split(X, y):
        est = clone(pipe)
        est.fit(X.iloc[tr], y.iloc[tr], lr__sample_weight=weights.iloc[tr])
        scores.append(average_precision_score(y.iloc[te], est.predict_proba(X.iloc[te])[:, 1]))
    return np.array(scores)

train_mask = df.year.between(2014, 2021) & df.made_it_contract.notna()
train_df   = df[train_mask].copy()
train_df['made_it_contract'] = train_df['made_it_contract'].astype(int)

block_train    = train_df[train_df['Pos_Group'].isin(BLOCK)].copy()
coverage_train = train_df[train_df['Pos_Group'].isin(COVERAGE)].copy()
skill_train    = train_df[train_df['Pos_Group'].isin(SKILL)].copy()
qb_train       = train_df[train_df['Pos_Group'].isin(QB_GRP)].copy()

y_block, y_coverage, y_skill, y_qb = (
    block_train['made_it_contract'],
    coverage_train['made_it_contract'],
    skill_train['made_it_contract'],
    qb_train['made_it_contract'],
)

for label, tr, y in [('Block   ', block_train, y_block),
                      ('Coverage', coverage_train, y_coverage),
                      ('Skill   ', skill_train, y_skill),
                      ('QB      ', qb_train, y_qb)]:
    print(f"{label} — {len(tr)} players, {int(y.sum())} positives ({y.mean():.1%})")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

block_pipe        = make_full_pipe();  block_baseline    = make_baseline_pipe()
coverage_pipe     = make_full_pipe();  coverage_baseline = make_baseline_pipe()
skill_pipe        = make_full_pipe();  skill_baseline    = make_baseline_pipe()
qb_pipe           = make_full_pipe();  qb_baseline       = make_baseline_pipe()

block_text_cv  = cross_val_score(block_pipe,    block_train,    y_block,    cv=cv, scoring='average_precision')
cov_text_cv    = cross_val_score(coverage_pipe, coverage_train, y_coverage, cv=cv, scoring='average_precision')
skill_text_cv  = cross_val_score(skill_pipe,    skill_train,    y_skill,    cv=cv, scoring='average_precision')
qb_text_cv     = cross_val_score(qb_pipe,       qb_train,       y_qb,       cv=cv, scoring='average_precision')

block_base_cv  = cv_baseline(block_baseline,    block_train,    y_block,    cv)
cov_base_cv    = cv_baseline(coverage_baseline, coverage_train, y_coverage, cv)
skill_base_cv  = cv_baseline(skill_baseline,    skill_train,    y_skill,    cv)
qb_base_cv     = cv_baseline(qb_baseline,       qb_train,       y_qb,       cv)

print("=== Cross-validated PR-AUC ===")
print(f"{'Group':<12} {'Baseline':>10}  {'Text+Grade':>10}  {'Delta':>8}")
for label, base_cv, text_cv in [
    ('Block',    block_base_cv, block_text_cv),
    ('Coverage', cov_base_cv,   cov_text_cv),
    ('Skill',    skill_base_cv, skill_text_cv),
    ('QB',       qb_base_cv,    qb_text_cv),
]:
    print(f"{label:<12} {base_cv.mean():>10.3f}  {text_cv.mean():>10.3f}  {text_cv.mean()-base_cv.mean():>+8.3f}")

In [ ]:
# Fit all models
block_pipe.fit(block_train, y_block)
coverage_pipe.fit(coverage_train, y_coverage)
skill_pipe.fit(skill_train, y_skill)
qb_pipe.fit(qb_train, y_qb)
fit_baseline(block_baseline,    block_train,    y_block)
fit_baseline(coverage_baseline, coverage_train, y_coverage)
fit_baseline(skill_baseline,    skill_train,    y_skill)
fit_baseline(qb_baseline,       qb_train,       y_qb)

oof_block    = cross_val_predict(block_pipe,    block_train,    y_block,    cv=cv, method='predict_proba')[:, 1]
oof_coverage = cross_val_predict(coverage_pipe, coverage_train, y_coverage, cv=cv, method='predict_proba')[:, 1]
oof_skill    = cross_val_predict(skill_pipe,    skill_train,    y_skill,    cv=cv, method='predict_proba')[:, 1]
oof_qb       = cross_val_predict(qb_pipe,       qb_train,       y_qb,       cv=cv, method='predict_proba')[:, 1]

def make_scatter(train, oof, base_pipe):
    s = train.copy()
    s['text_score']     = oof
    s['baseline_score'] = base_pipe.predict_proba(train)[:, 1]
    return s

future_df = df[df.year.between(2022, 2025)].copy()

def append_future(scatter, pos_set, text_pipe, base_pipe):
    fdf = future_df[future_df['Pos_Group'].isin(pos_set)].copy()
    fdf['text_score']     = text_pipe.predict_proba(fdf)[:, 1]
    fdf['baseline_score'] = base_pipe.predict_proba(fdf)[:, 1]
    return pd.concat([scatter, fdf], ignore_index=True)

scatter_block    = append_future(make_scatter(block_train,    oof_block,    block_baseline),    BLOCK,    block_pipe,    block_baseline)
scatter_coverage = append_future(make_scatter(coverage_train, oof_coverage, coverage_baseline), COVERAGE, coverage_pipe, coverage_baseline)
scatter_skill    = append_future(make_scatter(skill_train,    oof_skill,    skill_baseline),    SKILL,    skill_pipe,    skill_baseline)
scatter_qb       = append_future(make_scatter(qb_train,       oof_qb,       qb_baseline),       QB_GRP,   qb_pipe,       qb_baseline)

print('Done.')

In [ ]:
# ── 4×2 scatter: grade vs baseline / grade vs text ────────────────────────────
groups = [
    ('Block (OL+DT)',       scatter_block,    BLOCK),
    ('Coverage (EDGE+DB+LB)', scatter_coverage, COVERAGE),
    ('Skill (WR+RB+TE)',    scatter_skill,    SKILL),
    ('QB',                  scatter_qb,       QB_GRP),
]

fig, axes = plt.subplots(4, 2, figsize=(12, 16), sharey='row')
CUTOFF = 0.30

for row_idx, (label, sc, _) in enumerate(groups):
    labeled   = sc[sc['made_it_contract'].notna()].copy()
    unlabeled = sc[sc['made_it_contract'].isna()].copy()
    colors_l  = labeled['year'].astype(int)
    cmap      = plt.cm.viridis

    for col_idx, score_col in enumerate(['baseline_score', 'text_score']):
        ax = axes[row_idx, col_idx]
        sc_lab = ax.scatter(labeled['grade'], labeled[score_col],
                            c=colors_l, cmap=cmap, alpha=0.55, s=15,
                            vmin=2014, vmax=2025, label='labeled')
        ax.scatter(unlabeled['grade'], unlabeled[score_col],
                   c='lightgray', alpha=0.35, s=10, label='2022-25')
        ax.axhline(CUTOFF, color='tomato', lw=1, ls='--')
        ax.set_title(f'{label}\n{["Baseline", "Text+Grade"][col_idx]}', fontsize=9)
        ax.set_xlabel('Grade'); ax.set_ylabel('Pred prob')
        ax.set_ylim(0, 1)
        if row_idx == 0 and col_idx == 1:
            cbar = fig.colorbar(sc_lab, ax=ax)
            cbar.set_label('Year')

plt.tight_layout()
plt.show()

In [ ]:
# ── Section contribution analysis ────────────────────────────────────────────
# Which section (ovr/str/wkn) drives the most RF importance per group?

def section_importance(pipe, label):
    text_pipeline  = pipe.named_steps['features'].transformer_list[0][1]
    tfidf          = text_pipeline.named_steps['tfidf']
    rf             = pipe.named_steps['rf']
    vocab          = {v: k for k, v in tfidf.vocabulary_.items()}
    n_text         = len(vocab)
    importances    = rf.feature_importances_[:n_text]

    sec_imp = {'ovr': 0.0, 'str': 0.0, 'wkn': 0.0, 'other': 0.0}
    for idx, imp in enumerate(importances):
        term = vocab[idx]
        prefix = term.split('_')[0] if '_' in term else 'other'
        if prefix in sec_imp:
            sec_imp[prefix] += imp
        else:
            sec_imp['other'] += imp

    total = sum(sec_imp.values()) or 1
    print(f"  {label}")
    for sec, imp in sorted(sec_imp.items(), key=lambda x: x[1], reverse=True):
        print(f"    {sec:<8} {imp/total:>6.1%}  (abs: {imp:.4f})")

print("=== RF Feature Importance by Section ===")
for pipe, label in [
    (block_pipe,    'Block (OL+DT)'),
    (coverage_pipe, 'Coverage (EDGE+DB+LB)'),
    (skill_pipe,    'Skill (WR+RB+TE)'),
    (qb_pipe,       'QB'),
]:
    section_importance(pipe, label)
    print()

In [ ]:
def _strip_prefixes(term):
    """Strip ovr_/str_/wkn_ prefix from each token in a unigram or bigram term."""
    parts = term.split(' ')
    cleaned = []
    for tok in parts:
        idx = tok.find('_')
        cleaned.append(tok[idx+1:] if idx != -1 else tok)
    return ' '.join(cleaned)

def _get_prefix(term):
    """Section prefix from the first token of a possibly-bigram term."""
    first = term.split(' ')[0]
    idx = first.find('_')
    return first[:idx] if idx != -1 else None

def top_section_features(pipe, label, n=15, show_raw=False):
    text_pipeline = pipe.named_steps['features'].transformer_list[0][1]
    tfidf         = text_pipeline.named_steps['tfidf']
    rf            = pipe.named_steps['rf']
    vocab         = {v: k for k, v in tfidf.vocabulary_.items()}
    n_text        = len(vocab)
    importances   = rf.feature_importances_[:n_text]

    by_section = {'ovr': [], 'str': [], 'wkn': []}
    for idx, imp in enumerate(importances):
        term   = vocab[idx]
        prefix = _get_prefix(term)
        if prefix in by_section:
            clean = _strip_prefixes(term)
            by_section[prefix].append((term, clean, imp))

    print(f"=== {label} — Top {n} features per section ===")
    for sec in ['str', 'wkn', 'ovr']:
        top = sorted(by_section[sec], key=lambda x: x[2], reverse=True)[:n]
        print(f"  [{sec.upper()}]  {'Term':<30} {'RF Importance':>14}{'  Raw term' if show_raw else ''}")
        for raw_term, clean, imp in top:
            raw_col = f'  ({raw_term})' if show_raw else ''
            print(f"         {clean:<30} {imp:>14.5f}{raw_col}")
        print()

for pipe, label in [
    (block_pipe,    'Block (OL+DT)'),
    (coverage_pipe, 'Coverage (EDGE+DB+LB)'),
    (skill_pipe,    'Skill (WR+RB+TE)'),
    (qb_pipe,       'QB'),
]:
    top_section_features(pipe, label)

In [ ]:
# ── Aaron Donald deep dive ───────────────────────────────────────────────────
TARGET = 'Aaron Donald'
ad_row = df[df['player_name'].str.lower() == TARGET.lower()]
if len(ad_row) == 0:
    ad_row = df[df['player_name'].str.contains('Donald', case=False)]
print(f"Found: {ad_row[['player_name','year','Pos_Group','grade','made_it_contract']].to_string(index=False)}\n")

ad  = ad_row.iloc[[0]]
ad2 = pd.concat([ad, ad], ignore_index=True)

baseline_prob = block_baseline.predict_proba(ad)[0, 1]
text_prob     = block_pipe.predict_proba(ad2)[0, 1]
print(f"Baseline score:   {baseline_prob:.3f}")
print(f"Text+Grade score: {text_prob:.3f}")
print(f"Text lift:        {text_prob - baseline_prob:+.3f}\n")

text_pipeline = block_pipe.named_steps['features'].transformer_list[0][1]
tfidf         = text_pipeline.named_steps['tfidf']
rf            = block_pipe.named_steps['rf']
vocab         = {v: k for k, v in tfidf.vocabulary_.items()}
n_text        = len(vocab)
importances   = rf.feature_importances_[:n_text]

ad_vec   = tfidf.transform(ad['scouting_text_tagged']).toarray()[0]
combined = ad_vec * importances

by_sec = {'ovr': [], 'str': [], 'wkn': []}
for idx in range(n_text):
    if ad_vec[idx] == 0:
        continue
    term   = vocab[idx]
    prefix = _get_prefix(term)
    if prefix in by_sec:
        clean = _strip_prefixes(term)
        by_sec[prefix].append((clean, term, ad_vec[idx], importances[idx], combined[idx]))

print("=== Donald's top 15 features by section (TF-IDF × RF importance) ===")
print("  clean term = display name | raw = internal prefixed token")
for sec in ['str', 'wkn', 'ovr']:
    top = sorted(by_sec[sec], key=lambda x: x[4], reverse=True)[:15]
    if not top:
        continue
    sec_total = sum(x[4] for x in by_sec[sec])
    print(f"\n  [{sec.upper()}]  section combined importance: {sec_total:.4f}")
    print(f"  {'Term':<25} {'Raw token':<35} {'TF-IDF':>8}  {'RF Imp':>8}  {'Combined':>10}")
    for clean, raw, tf, rf_imp, comb in top:
        print(f"  {clean:<25} {raw:<35} {tf:>8.4f}  {rf_imp:>8.5f}  {comb:>10.6f}")

print("\n=== Which section drove Donald's text score? ===")
for sec in ['str', 'wkn', 'ovr']:
    total  = sum(x[4] for x in by_sec[sec])
    n_terms = len(by_sec[sec])
    print(f"  {sec.upper()}: combined importance {total:.4f}  ({n_terms} active terms)")

In [ ]:
# ── Section attribution scatter ─────────────────────────────────────────────
# For each player: what % of their text score comes from STR / WKN / OVR?
from scipy import stats as sp_stats
from scipy.sparse import issparse

def compute_section_attribution(pipe, df_grp):
    text_pipeline = pipe.named_steps['features'].transformer_list[0][1]
    tfidf         = text_pipeline.named_steps['tfidf']
    rf            = pipe.named_steps['rf']
    vocab         = {v: k for k, v in tfidf.vocabulary_.items()}
    n_text        = len(vocab)
    importances   = rf.feature_importances_[:n_text]

    X_text = tfidf.transform(df_grp['scouting_text_tagged'].fillna(''))
    if issparse(X_text):
        X_text = X_text.toarray()
    weighted = X_text * importances[np.newaxis, :]

    sec_masks = {'ovr': np.zeros(n_text, bool),
                 'str': np.zeros(n_text, bool),
                 'wkn': np.zeros(n_text, bool)}
    for idx in range(n_text):
        p = _get_prefix(vocab[idx])
        if p in sec_masks:
            sec_masks[p][idx] = True

    total = weighted.sum(axis=1)
    total = np.where(total == 0, 1, total)

    out = df_grp[['player_name', 'Pos_Group', 'grade', 'made_it_contract']].copy().reset_index(drop=True)
    for sec, mask in sec_masks.items():
        out[f'{sec}_pct'] = weighted[:, mask].sum(axis=1) / total * 100
    out['text_score'] = pipe.predict_proba(df_grp)[:, 1]
    return out

attr_parts = []
for pipe, df_grp in [
    (block_pipe,    block_train),
    (coverage_pipe, coverage_train),
    (skill_pipe,    skill_train),
    (qb_pipe,       qb_train),
]:
    attr_parts.append(compute_section_attribution(pipe, df_grp))
attr_df = pd.concat(attr_parts, ignore_index=True)

attr_df['grade_bucket'] = pd.cut(
    attr_df['grade'],
    bins=[0, 5.85, 6.05, 10],
    labels=['Low (<5.85)', 'Mid (5.85–6.05)', 'High (>6.05)']
)

bucket_colors = {'Low (<5.85)': '#FF9800', 'Mid (5.85–6.05)': '#9C27B0', 'High (>6.05)': '#2196F3'}
sections = [('str', 'Strengths %', '#1565C0'),
            ('wkn', 'Weaknesses %', '#B71C1C'),
            ('ovr', 'Overview %',   '#1B5E20')]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, (sec, sec_label, line_color) in zip(axes, sections):
    for bucket, color in bucket_colors.items():
        mask = attr_df['grade_bucket'] == bucket
        ax.scatter(
            attr_df.loc[mask, 'text_score'],
            attr_df.loc[mask, f'{sec}_pct'],
            c=color, alpha=0.35, s=14, label=bucket, zorder=2
        )
    # regression line
    x = attr_df['text_score'].values
    y = attr_df[f'{sec}_pct'].values
    slope, intercept, r, pval, _ = sp_stats.linregress(x, y)
    xs = np.linspace(x.min(), x.max(), 200)
    ax.plot(xs, slope * xs + intercept, color=line_color, lw=2,
            label=f'r={r:.2f} (p={pval:.3f})')
    ax.set_xlabel('Text score (predicted prob)', fontsize=10)
    ax.set_ylabel(f'% importance from {sec_label}', fontsize=10)
    ax.set_title(sec_label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.25)

plt.suptitle('Section attribution vs text score — training set, colored by grade bucket',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Also: summary table
print("\nMean section attribution by grade bucket:")
print(attr_df.groupby('grade_bucket', observed=True)[['str_pct','wkn_pct','ovr_pct']].mean().round(1).to_string())
print("\nMean section attribution by Pos_Group:")
print(attr_df.groupby('Pos_Group')[['str_pct','wkn_pct','ovr_pct']].mean().round(1).to_string())

In [ ]:
# ── Word clouds: success vs failure terms per position group ─────────────────
# Terms sized by RF_importance × (mean_TFIDF_made_it1 - mean_TFIDF_made_it0)
# Color by section: STR=blue, WKN=red, OVR=green
from wordcloud import WordCloud
from scipy.sparse import issparse

SEC_COLORS = {'str': '#1565C0', 'wkn': '#C62828', 'ovr': '#2E7D32'}

def compute_term_weights(pipe, df_grp, y):
    text_pipeline = pipe.named_steps['features'].transformer_list[0][1]
    tfidf         = text_pipeline.named_steps['tfidf']
    rf            = pipe.named_steps['rf']
    vocab         = {v: k for k, v in tfidf.vocabulary_.items()}
    n_text        = len(vocab)
    importances   = rf.feature_importances_[:n_text]

    X_text = tfidf.transform(df_grp['scouting_text_tagged'].fillna(''))
    if issparse(X_text):
        X_text = X_text.toarray()

    pos_mask = y.values == 1
    neg_mask = ~pos_mask
    mean_pos = X_text[pos_mask].mean(axis=0) if pos_mask.sum() > 0 else np.zeros(n_text)
    mean_neg = X_text[neg_mask].mean(axis=0) if neg_mask.sum() > 0 else np.zeros(n_text)
    diff = (mean_pos - mean_neg) * importances

    success, failure, prefixes = {}, {}, {}
    for idx in range(n_text):
        term    = vocab[idx]
        prefix  = _get_prefix(term)
        display = _strip_prefixes(term).replace(' ', '_')   # bigrams: space→underscore
        prefixes[display] = prefix
        if diff[idx] > 0:
            success[display] = float(diff[idx])
        elif diff[idx] < 0:
            failure[display] = float(-diff[idx])

    return success, failure, prefixes

def make_color_func(prefixes):
    def color_func(word, **kwargs):
        return SEC_COLORS.get(prefixes.get(word, 'ovr'), '#555555')
    return color_func

groups_wc = [
    (block_pipe,    block_train,    y_block,    'Block (OL + DT)'),
    (coverage_pipe, coverage_train, y_coverage, 'Coverage (EDGE + DB + LB)'),
    (skill_pipe,    skill_train,    y_skill,    'Skill (WR + RB + TE)'),
    (qb_pipe,       qb_train,       y_qb,       'QB'),
]

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
fig.patch.set_facecolor('#FAFAFA')

for row, (pipe, df_grp, y, label) in enumerate(groups_wc):
    success, failure, prefixes = compute_term_weights(pipe, df_grp, y)
    cf = make_color_func(prefixes)

    for col, (weights, title, bg, border) in enumerate([
        (success, f'SUCCESS  ·  {label}', '#EEF4FF', '#1565C0'),
        (failure, f'FAILURE  ·  {label}', '#FFF0F0', '#C62828'),
    ]):
        ax = axes[row][col]
        ax.set_facecolor(bg)
        if weights:
            wc = WordCloud(
                width=700, height=320,
                background_color=bg,
                max_words=60,
                prefer_horizontal=0.80,
                color_func=cf,
                collocations=False,
            ).generate_from_frequencies(weights)
            ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold', pad=8,
                     color=border)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border)
            spine.set_linewidth(1.5)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=c, label=s.upper())
              for s, c in SEC_COLORS.items()]
fig.legend(handles=legend_els, loc='lower center', ncol=3,
           fontsize=10, title='Section', title_fontsize=10,
           frameon=True, bbox_to_anchor=(0.5, 0.005))

plt.suptitle('Top discriminating terms by position group\n'
             'Sized by RF importance × TF-IDF differential (made_it=1 vs 0)',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

In [ ]:
# ── Text lift bar chart + sleeper scatter ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── LEFT: horizontal delta bars ───────────────────────────────────────────────
grp_labels  = ['Block\n(OL+DT)', 'Coverage\n(EDGE+DB+LB)', 'Skill\n(WR+RB+TE)', 'QB']
base_scores = [block_base_cv.mean(), cov_base_cv.mean(), skill_base_cv.mean(), qb_base_cv.mean()]
text_scores = [block_text_cv.mean(), cov_text_cv.mean(), skill_text_cv.mean(), qb_text_cv.mean()]
deltas      = [t - b for t, b in zip(text_scores, base_scores)]
bar_colors  = ['#1565C0' if d > 0 else '#C62828' for d in deltas]

ax = axes[0]
bars = ax.barh(grp_labels, deltas, color=bar_colors, alpha=0.85, height=0.45, zorder=2)
ax.axvline(0, color='black', lw=1.2, zorder=3)
ax.grid(axis='x', alpha=0.3, zorder=1)

for bar, delta, b, t in zip(bars, deltas, base_scores, text_scores):
    pad = 0.001
    ha  = 'left' if delta >= 0 else 'right'
    x   = delta + pad if delta >= 0 else delta - pad
    ax.text(x, bar.get_y() + bar.get_height() / 2,
            f'{delta:+.3f}  ({b:.3f}→{t:.3f})',
            va='center', ha=ha, fontsize=9.5)

ax.set_xlabel('PR-AUC delta  (text model − baseline)', fontsize=11)
ax.set_title('Text model lift by position group\n(5-fold CV, PR-AUC)', fontsize=12, fontweight='bold')
ax.set_xlim(min(deltas) - 0.06, max(deltas) + 0.09)

# ── RIGHT: grade vs text_score scatter, highlight sleepers ───────────────────
# Combine all training groups with their OOF scores
all_parts = []
for df_grp, oof, grp_name in [
    (block_train,    oof_block,    'Block'),
    (coverage_train, oof_coverage, 'Coverage'),
    (skill_train,    oof_skill,    'Skill'),
    (qb_train,       oof_qb,       'QB'),
]:
    tmp = df_grp[['player_name', 'Pos_Group', 'grade', 'made_it_contract']].copy().reset_index(drop=True)
    tmp['text_score'] = oof
    tmp['group']      = grp_name
    all_parts.append(tmp)
scatter_all = pd.concat(all_parts, ignore_index=True)

SLEEPER_GRADE = 5.9
SLEEPER_THRESH = 0.35

GRP_COLORS = {'Block': '#1565C0', 'Coverage': '#2E7D32', 'Skill': '#F57F17', 'QB': '#6A1B9A'}

ax2 = axes[1]
for grp, color in GRP_COLORS.items():
    m = scatter_all['group'] == grp
    made = m & (scatter_all['made_it_contract'] == 1)
    miss = m & (scatter_all['made_it_contract'] == 0)
    ax2.scatter(scatter_all.loc[miss, 'grade'], scatter_all.loc[miss, 'text_score'],
                c=color, alpha=0.15, s=18, marker='o')
    ax2.scatter(scatter_all.loc[made, 'grade'], scatter_all.loc[made, 'text_score'],
                c=color, alpha=0.55, s=28, marker='o', label=grp)

# Sleeper quadrant box
ax2.axvline(SLEEPER_GRADE,  color='gray', lw=1, ls='--', alpha=0.7)
ax2.axhline(SLEEPER_THRESH, color='gray', lw=1, ls='--', alpha=0.7)
ax2.fill_betweenx([SLEEPER_THRESH, 1.0], 0, SLEEPER_GRADE,
                  alpha=0.07, color='gold', label='Sleeper zone')

# Label notable sleepers (low grade, high text score, made_it=1)
sleepers_plot = scatter_all[
    (scatter_all['grade'] < SLEEPER_GRADE) &
    (scatter_all['text_score'] >= SLEEPER_THRESH) &
    (scatter_all['made_it_contract'] == 1)
].nlargest(10, 'text_score')

for _, row in sleepers_plot.iterrows():
    ax2.annotate(
        row['player_name'].split()[-1],   # last name only
        xy=(row['grade'], row['text_score']),
        xytext=(4, 2), textcoords='offset points',
        fontsize=6.5, alpha=0.85,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
    )

ax2.set_xlabel('Scout grade', fontsize=11)
ax2.set_ylabel('Text model OOF score', fontsize=11)
ax2.set_title('Grade vs text score (OOF)\nFull dots = made second contract', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8, loc='upper left', title='Group (bright=made_it)')
ax2.set_xlim(4.5, 8.0)
ax2.set_ylim(-0.02, 1.02)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Residual plot: who did the text model push across the threshold? ──────────
# x = baseline score, y = text OOF score
# y=x diagonal = no lift from text
# Threshold lines divide the space into quadrants:
#   Top-left   (base<T, text>=T): text boosted above threshold
#   Bottom-right (base>=T, text<T): text pushed below threshold
# Color = actual outcome. Annotate players who crossed AND got it right.

THRESH = 0.3

# Rebuild combined scatter (OOF for text, in-sample for baseline)
_parts = []
for df_grp, oof, base_pipe, grp in [
    (block_train,    oof_block,    block_baseline,    'Block'),
    (coverage_train, oof_coverage, coverage_baseline, 'Coverage'),
    (skill_train,    oof_skill,    skill_baseline,    'Skill'),
    (qb_train,       oof_qb,       qb_baseline,       'QB'),
]:
    tmp = df_grp[['player_name', 'Pos_Group', 'grade', 'made_it_contract']].copy().reset_index(drop=True)
    tmp['text_oof']  = oof
    tmp['base_score'] = base_pipe.predict_proba(df_grp)[:, 1]
    tmp['residual']   = tmp['text_oof'] - tmp['base_score']
    tmp['group']      = grp
    _parts.append(tmp)
resid_df = pd.concat(_parts, ignore_index=True)

# Classify each player into crossing quadrant
resid_df['crossed_up']   = (resid_df['base_score'] < THRESH) & (resid_df['text_oof'] >= THRESH)
resid_df['crossed_down'] = (resid_df['base_score'] >= THRESH) & (resid_df['text_oof'] < THRESH)
resid_df['crossed']      = resid_df['crossed_up'] | resid_df['crossed_down']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

OUTCOME_COLORS = {1: '#E65100', 0: '#546E7A'}   # orange = made it, slate = didn't

# ── LEFT: full scatter ────────────────────────────────────────────────────────
ax = axes[0]
for made, color, label, alpha, size in [
    (0, '#90A4AE', "Didn't make it", 0.25, 18),
    (1, '#E65100', 'Made it',        0.60, 28),
]:
    m = resid_df['made_it_contract'] == made
    ax.scatter(resid_df.loc[m, 'base_score'], resid_df.loc[m, 'text_oof'],
               c=color, alpha=alpha, s=size, label=label, zorder=2)

# y=x line
lims = [0, 1]
ax.plot(lims, lims, 'k--', lw=1.2, alpha=0.5, label='y = x (no lift)', zorder=3)

# Threshold lines + quadrant shading
ax.axvline(THRESH, color='steelblue', lw=1.2, ls=':', alpha=0.8)
ax.axhline(THRESH, color='steelblue', lw=1.2, ls=':', alpha=0.8)

ax.fill_between([0, THRESH], THRESH, 1.0, alpha=0.07, color='gold',
                label=f'Boosted above {THRESH}')
ax.fill_between([THRESH, 1.0], 0, THRESH, alpha=0.07, color='red',
                label=f'Pushed below {THRESH}')

# Annotate top boosted true positives (crossed_up, made_it=1)
boosted_tp = resid_df[resid_df['crossed_up'] & (resid_df['made_it_contract'] == 1)].nlargest(12, 'residual')
for _, row in boosted_tp.iterrows():
    ax.annotate(row['player_name'].split()[-1],
                xy=(row['base_score'], row['text_oof']),
                xytext=(5, 2), textcoords='offset points',
                fontsize=6.5, color='#E65100',
                arrowprops=dict(arrowstyle='-', color='#E65100', lw=0.5))

# Annotate top penalised true negatives (crossed_down, made_it=0)
penalised_tn = resid_df[resid_df['crossed_down'] & (resid_df['made_it_contract'] == 0)].nsmallest(8, 'residual')
for _, row in penalised_tn.iterrows():
    ax.annotate(row['player_name'].split()[-1],
                xy=(row['base_score'], row['text_oof']),
                xytext=(5, -8), textcoords='offset points',
                fontsize=6.5, color='#546E7A',
                arrowprops=dict(arrowstyle='-', color='#546E7A', lw=0.5))

ax.set_xlabel('Baseline score (grade + position)', fontsize=11)
ax.set_ylabel('Text model OOF score', fontsize=11)
ax.set_title(f'Baseline vs text score — threshold crossers (T={THRESH})\n'
             f'Orange names = true sleepers found  |  Gray names = busts penalised', fontsize=10.5)
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.2)

# ── RIGHT: residual bar chart, top crossers ────────────────────────────────────
ax2 = axes[1]

# Top 20 boosted (crossed_up, any outcome) by residual
up   = resid_df[resid_df['crossed_up']].nlargest(20, 'residual').copy()
# Top 15 dropped (crossed_down, any outcome) by residual magnitude
down = resid_df[resid_df['crossed_down']].nsmallest(15, 'residual').copy()

combined = pd.concat([up, down]).sort_values('residual', ascending=True)
bar_colors = combined.apply(
    lambda r: '#E65100' if (r['crossed_up']   and r['made_it_contract'] == 1) else
              '#FFB300' if (r['crossed_up']   and r['made_it_contract'] == 0) else
              '#546E7A' if (r['crossed_down'] and r['made_it_contract'] == 0) else
              '#EF5350',   # crossed_down but made_it=1 (text wrongly hurt them)
    axis=1
)
labels = combined['player_name'] + ' (' + combined['Pos_Group'] + ')'

ax2.barh(range(len(combined)), combined['residual'], color=bar_colors.values, alpha=0.85, height=0.7)
ax2.set_yticks(range(len(combined)))
ax2.set_yticklabels(labels, fontsize=7.5)
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('Text residual  (text OOF − baseline)', fontsize=11)
ax2.set_title('Threshold crossers ranked by residual magnitude\n'
              'Orange=true sleeper  Yellow=false alarm  Gray=true bust penalised  Red=wrongly hurt', fontsize=10)
ax2.grid(axis='x', alpha=0.25)

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#E65100', label='Boosted → made it (true sleeper)'),
    Patch(facecolor='#FFB300', label="Boosted → didn't make it (false alarm)"),
    Patch(facecolor='#546E7A', label="Dropped → didn't make it (true bust caught)"),
    Patch(facecolor='#EF5350', label='Dropped → made it (wrongly penalised)'),
]
ax2.legend(handles=legend_els, fontsize=7.5, loc='lower right')

plt.tight_layout()
plt.show()

# Summary counts
print(f"Threshold = {THRESH}")
print(f"Boosted above threshold:  {resid_df['crossed_up'].sum()} players")
print(f"  → made it (true sleepers):  {(resid_df['crossed_up'] & (resid_df['made_it_contract']==1)).sum()}")
print(f"  → didn't (false alarms):    {(resid_df['crossed_up'] & (resid_df['made_it_contract']==0)).sum()}")
print(f"Pushed below threshold:   {resid_df['crossed_down'].sum()} players")
print(f"  → didn't (correct):         {(resid_df['crossed_down'] & (resid_df['made_it_contract']==0)).sum()}")
print(f"  → made it (wrong):          {(resid_df['crossed_down'] & (resid_df['made_it_contract']==1)).sum()}")